# MindFlow Step 3a · 5-minute charts, volatility target

**Question.** Same charts and the same models as Step 2, but the target is the size of the move: is |r(next h)| above the stock's own training-period median? Is there a signal, and which model wins?


Why volatility: returns are r = μ + σ·ε with ε unpredictable, so direction is a coin flip, but σ is persistent
(volatility clustering, ARCH/GARCH) and has an intraday U-shape, so the size of the next move is predictable. The
threshold is each stock's own median, so volatile stocks do not win by construction.


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test) |
| **7** | Efficient computing for the final model: 16-bit / INT8 inference, LoRA fine-tuning of a pretrained ViT | `Step7a_Efficient_Computing` |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

**Models** (same rows for every model; parameter counts printed in section 5 and saved per cell):

| model | input | parameters |
|---|---|---|
| `plain` image CNN | 234×234 chart of the stock's 78 bars | 776,450 |
| `plain_spy` image CNN + SPY | 468×234: stock chart above SPY's chart, bar columns aligned | 936,194 |
| `residual` ResNet-style image CNN | 234×234 chart | 2,047,362 |
| `series1d` 1D CNN | the 5 OHLCV series, scaled exactly as the chart is | 720,770 |
| `series1d_spy` 1D CNN + SPY | the stock's 5 series and SPY's 5 | 723,970 |
| `vit` image ViT (Tokens-to-Token style, from scratch) | 234×234 chart, 169 tokens, 8 blocks of width 96 | 864,002 |
| `vit_spy` image ViT + SPY | 468×234 (338 tokens) | 880,226 |
| HGB hand features | 20 numbers of the same 78 bars (returns, realized vol, volume, time of day) | tree nodes |
| HGB hand + SPY | the 20 plus 18 SPY / stock-vs-SPY numbers | tree nodes |
| rules | last-bar return, 78-bar momentum (direction); mean recent |r| (volatility) | 0 |

The CNNs use Adam 1e-5 as in Jiang–Kelly–Xiu; the ViTs need their own rate (1e-4). Pretrained models (frozen DINOv2 /
CLIP, time-series foundation models) are in the `b` notebook of this step, on the same rows.

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
import chartcnn
from chartcnn import grid
from chartcnn.utils import save_json
assert chartcnn.__version__ >= "2026-11-23.1", f"stale chartcnn {chartcnn.__version__} in {CODE}"
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "")

## 2. Configuration
**To resume after a disconnect**, set `RUN_ID` to the id printed by the first run and Run all again: finished cells
are skipped and an unfinished network resumes from its last epoch. To add the 15- and 30-minute cells later, set
`HORIZONS = [1, 12, 3, 6]` with the same `RUN_ID`.

**To retrain only some networks of a finished run** (e.g. the ViTs after the 2026-11-23 fix), set `RUN_ID` to that run
and `RETRAIN_ARCHS = ["vit", "vit_spy"]`, keep every other setting, and Run all. Only those networks train again, on
exactly the same rows (checked); HGB, rules and the CNNs are kept, the old networks are moved to
`<cell>/<arch>_before_<version>/`, and the tables below are rebuilt with the new scores.

**To add the VIXY panel (intraday VIX proxy) to a finished run**: upload `vixy_panel_v4.zip` (built on the Mac with
`MindFlow/tools/build_panel_bundle.py`) next to the bundle in `Demo/MindFlow`, set `RUN_ID` to that run and
`RETRAIN_ARCHS = VIXY_ARCHS`, and Run all. The networks in `VIXY_ARCHS` (default: CNN and ViT with SPY + VIXY panels, three
stacked charts, 702 × 234) are trained on the same rows and added to the tables, next to their `_spy` versions.

**`QUICK = True`** is a small first pass (40,000 training windows, 3 epochs, every 5th validation day) to check that
everything runs; its numbers are rough.

**Time (estimate on an L4, not measured on v4):** 10–20 min per image CNN or ViT and 3–6 min per 1D CNN per cell,
so roughly 1–1.5 hours per cell for the seven networks, 2 cells ≈ 2.5 hours.
The networks train on a random 300,000 training windows (`max_train_nn`); HGB uses up to 1,000,000. Early stopping
watches every 4th validation day; the reported validation scores use all of 2023.

In [ ]:
SMOKE = LOCAL_TEST                                # tiny code check on synthetic data; never a result
assert DEVICE.type == "cuda" or SMOKE, "Select a GPU runtime"
RUN_ID = None                                     # None = new run; or an old RUN_ID to resume / extend it
QUICK = False
SETTINGS = dict(
    learning_rate=1e-5, weight_decay=0.0,         # Adam, as in Jiang-Kelly-Xiu
    batch_size=4 if SMOKE else 128, eval_batch_size=8 if SMOKE else 1024,
    epochs=1 if SMOKE else 8, patience=2, min_delta=1e-5,       # early stop on validation log loss
    channels=[64, 128, 256], head_height=4, dropout=0.5,
    vit_learning_rate=3e-4, vit_weight_decay=0.05, vit_warmup_steps=1000, vit_grad_clip=1.0,   # AdamW
    vit_pool="mean", vit_dim=96, vit_depth=8, vit_heads=4, vit_mlp_ratio=3.0, vit_drop_path=0.1, vit_dropout=0.1,
    max_train_nn=300_000, max_train_hgb=1_000_000, early_stop_every_kth_day=4,
    amp=True,                                     # bfloat16 autocast on A100/L4/H100; ignored on T4/CPU
    num_workers=0 if SMOKE else min(8, os.cpu_count() or 2),
    seed=69,
)
if QUICK:
    SETTINGS.update(max_train_nn=40_000, max_train_hgb=200_000, validation_every_kth_day=5, epochs=3)
HORIZONS = [1, 12]                             # bars of 5 minutes (1 = 5 min, 12 = 60 min); add 3, 6 later
KINDS = ["vol"]
STRIDE = 12                                       # one sample per stock per hour, for every horizon
ARCHS = ['plain', 'plain_spy', 'residual', 'series1d', 'series1d_spy', 'vit', 'vit_spy']
RETRAIN_ARCHS = []                                # e.g. ["vit", "vit_spy"] with an old RUN_ID: retrain only these
VIXY_ARCHS = ["plain_spy_vixy", "vit_spy_vixy"]   # also possible: "plain_vixy", "vit_vixy"
if SMOKE:
    ARCHS = ["plain", "plain_spy", "series1d", "vit"]
    VIXY_ARCHS = ["plain_spy_vixy", "vit_vixy"]
if "vixy" in PANELS_ADDED:                        # vixy_panel_v4.zip is in Demo/MindFlow
    ARCHS = ARCHS + [a for a in VIXY_ARCHS if a not in ARCHS]
CELLS = grid.grid_cells(HORIZONS, KINDS, STRIDE)
RUN_ID, RUN_DRIVE, RESUME = mf.new_run(HERE, "3a", RUN_ID, "grid", LOCAL_TEST)
RUN_LOCAL = LOCAL_ROOT / "work" / RUN_DRIVE.parent.name / RUN_ID
RUN_LOCAL.mkdir(parents=True, exist_ok=True)
mf.save_settings(RUN_DRIVE, RESUME, settings=SETTINGS, cells=CELLS, archs=ARCHS, retrain=RETRAIN_ARCHS, quick=QUICK,
                 chartcnn=chartcnn.__version__)
print("RUN_ID:", RUN_ID, "->", RUN_DRIVE)
print(len(CELLS), "cells:", ", ".join(c["name"] for c in CELLS), "|", len(ARCHS), "networks:", ", ".join(ARCHS))

## 3. Data overview
Sessions, securities and windows per split as built and audited locally (`manifest.json`).

In [ ]:
store = chartcnn.Store(PREPARED)
M = json.loads((PREPARED / "manifest.json").read_text())
print("stocks:", len(M["permnos"]), "| slots:", store.T, "| sessions:", store.T // 78,
      "| early-close sessions excluded:", len(M["config"].get("early_close_sessions_excluded", [])))
print("SPY in bundle:", store.has_market, "| SPY valid slots:", M["bundle"]["market_valid_slots"])
print("windows in bundle:", M["bundle"]["windows"])
display(pd.DataFrame(M["splits"]).T[["samples", "decision_start", "decision_end"]])

## 4. What the models see
Training samples drawn by the same dataset code the models use: the stock chart alone, and with SPY's panel below.

In [ ]:
import matplotlib.pyplot as plt
sample, _ = chartcnn.build_split(store, "train", "vol", 1, STRIDE)
show = sample.sample(4, random_state=1).reset_index(drop=True)
plain, with_spy = chartcnn.make_dataset(PREPARED, show, "plain"), chartcnn.make_dataset(PREPARED, show, "plain_spy")
fig, axes = plt.subplots(2, 4, figsize=(16, 9), gridspec_kw=dict(height_ratios=[1, 2]))
for i in range(4):
    axes[0, i].imshow(plain.images(i), cmap="gray", aspect="auto")
    axes[1, i].imshow(with_spy.images(i), cmap="gray", aspect="auto")
    axes[0, i].set_title(f"PERMNO {show.permno[i]}  {store.dates[show.end_index[i]]}")
    for ax in axes[:, i]: ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel("plain (234x234)"); axes[1, 0].set_ylabel("plain_spy (468x234): stock / SPY")
fig.tight_layout(); fig.savefig(RUN_LOCAL / "what_models_see.png", dpi=110); plt.show()
shutil.copy2(RUN_LOCAL / "what_models_see.png", RUN_DRIVE / "what_models_see.png")

## 5. Models and parameter counts
Layer-by-layer output shapes and parameters. Output layers start at zero, so every initial prediction is exactly 0.5.

In [ ]:
PARAM_ROWS = []
for arch in ARCHS:
    data = chartcnn.make_dataset(PREPARED, show, arch)
    x = torch.stack([data[i][0] for i in range(4)])
    model = chartcnn.build_model(grid.arch_settings(SETTINGS, arch))
    table, total, trainable = chartcnn.parameter_table(model)
    print(f"{arch} ({grid.ARCH_LABEL[arch]}): input {tuple(x.shape[1:])}, {total:,} parameters ({trainable:,} trainable)")
    display(table)
    print("initial P(y=1):", model.eval()(x).softmax(1)[:, 1].tolist(), "\n")
    PARAM_ROWS.append(dict(model=grid.ARCH_LABEL[arch], arch=arch, input=str(tuple(x.shape[1:])), parameters=total))
del model
PARAMS = pd.DataFrame(PARAM_ROWS).set_index("arch")
PARAMS.to_csv(RUN_DRIVE / "network_parameters.csv"); display(PARAMS)

## 6. Labels
Sample counts and positive shares per cell (train 2016–2022, validation 2023).

In [ ]:
OVERVIEW = grid.label_overview(store, CELLS)
display(OVERVIEW.round(3)); OVERVIEW.to_csv(RUN_LOCAL / "labels.csv")

## 7. Train the grid (validation 2023)
Per cell: labels → rules and both HGBs → each network in `ARCHS` → validation predictions in
`runs/<step>/<RUN_ID>/<cell>/`. Watch `val AUC`, the log-loss gain vs the prior and `prob std` (≈ 0 = collapsed).

In [ ]:
RESULTS, t0 = {}, time.perf_counter()
for i, cell in enumerate(CELLS, 1):
    print("\n" + "=" * 20, f"[{i}/{len(CELLS)}]", cell["name"], "=" * 20)
    RESULTS[cell["name"]] = grid.run_cell(PREPARED, cell, SETTINGS, ARCHS, DEVICE, RUN_LOCAL / cell["name"],
                                          RUN_DRIVE / cell["name"], smoke=SMOKE, retrain=RETRAIN_ARCHS)
    print(f"elapsed {(time.perf_counter() - t0) / 60:.1f} min")

### Validation analysis
AUC per cell and model with 95% intervals from resampling whole validation days; paired comparisons (with vs without
SPY, ViT vs CNN, image vs 1D, CNN vs HGB) use the same resampled days for both models.

In [ ]:
TABLE, BOOTS = grid.grid_table(RESULTS, n_boot=50 if SMOKE else 300)
TABLE.to_csv(RUN_LOCAL / "grid_table.csv")
for model in ["CNN (plain)", "CNN (plain_spy)", "CNN (residual)", "ViT (vit)", "ViT (vit_spy)", "CNN (series1d)",
              "CNN (series1d_spy)", "HGB hand features", "HGB hand + SPY"]:
    if model in TABLE.index.get_level_values("model"):
        print(model); display(grid.auc_matrix(TABLE, model))
display(TABLE.drop(columns=["kind", "horizon"]).round(4))
grid.plot_grid(TABLE, RUN_LOCAL / "auc_by_horizon.png")
TAKEAWAYS = grid.takeaways(TABLE, BOOTS)
(RUN_LOCAL / "takeaways.md").write_text(TAKEAWAYS + "\n"); display(Markdown(TAKEAWAYS))
for name in ("labels.csv", "grid_table.csv", "auc_by_horizon.png", "takeaways.md"):
    shutil.copy2(RUN_LOCAL / name, RUN_DRIVE / name)
print("saved to", RUN_DRIVE)

### How to read it
- **All AUCs clearly above 0.5**: the size of the next move is predictable from the chart (volatility clustering
  plus the intraday U-shape). This is also the positive control: the pipeline can learn.
- **HGB vs image CNN** (paired line in the takeaways): on v1 data HGB won by 0.02–0.03 in every cell. Step 4 asks why.
- **ViT vs CNN**: the first v4 run's scratch ViT collapsed (constant output, AUC 0.500) because its tokens carried
  almost no image information at initialisation; fixed in chartcnn 2026-11-23.2 (fan-in init and LayerNorm in the
  tokenizer, mean pooling, AdamW with warmup and clipping, float32 head). Frozen pretrained ViTs are in Step 3b.

## 8. Held-out test 2024–2025 (run once, at the end)
Only after every choice above is final. Each network is scored from its best validation checkpoint; HGB is refit on the
same training rows. Set `RUN_TEST = True` and run this cell. Do not change anything after looking at it.

In [ ]:
RUN_TEST = SMOKE                                  # set True once, at the very end
if RUN_TEST:
    TEST = {}
    for cell in CELLS:
        folder = (RUN_LOCAL if SMOKE else RUN_DRIVE) / cell["name"]
        TEST[cell["name"]] = grid.evaluate_cell_test(PREPARED, cell, SETTINGS, ARCHS, DEVICE, folder, smoke=SMOKE)
    TEST_TABLE, TEST_BOOTS = grid.grid_table(TEST, n_boot=50 if SMOKE else 300)
    TEST_TABLE.to_csv(RUN_LOCAL / "test_grid_table.csv")
    display(TEST_TABLE.drop(columns=["kind", "horizon"]).round(4))
    display(Markdown(grid.takeaways(TEST_TABLE, TEST_BOOTS)))
    shutil.copy2(RUN_LOCAL / "test_grid_table.csv", RUN_DRIVE / "test_grid_table.csv")